# Citadel — Colab Training (Pipelines A & B)

Trains the 3 deep-learning models on **real curated data** and downloads the weights:
- **BehaveFormer + Siamese** → keystroke verification (Aalto) — metric: **EER**
- **CNN-LSTM** → mouse session anomaly (Balabit) — metric: **ROC-AUC**

**Before running:** `Runtime → Change runtime type → T4 GPU`.

You will upload **one zip** built locally with:
```bash
cd ml && zip -r citadel_train.zip src eval data/keystroke_subset.npz data/mouse_subset.npz
```
That zip is ~80 MB (the curated subsets, NOT the 18 GB raw data).

> The transaction-fraud models (XGBoost + Isolation Forest on PaySim) train
> locally on CPU via `python ml/src/train_fraud.py` — they are **not** in this notebook.

In [ ]:
# 1. Install dependencies
!pip -q install torch einops xgboost shap scikit-learn matplotlib
import torch; print("CUDA available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 2. Upload citadel_train.zip (src/ + eval/ + the two .npz subsets) and unzip
import os, zipfile
from google.colab import files
up = files.upload()                       # pick citadel_train.zip
zname = next(iter(up))
os.makedirs('ml', exist_ok=True)
with zipfile.ZipFile(zname) as z: z.extractall('ml')
print('Extracted to ml/:', os.listdir('ml'))
assert os.path.exists('ml/data/keystroke_subset.npz'), 'keystroke_subset.npz missing from zip'
assert os.path.exists('ml/data/mouse_subset.npz'), 'mouse_subset.npz missing from zip'

In [ ]:
# 3. Train all 3 DL models on GPU (best-accuracy settings ~ a few hours)
#    Drop --users to 5000 or --kp-epochs lower for a faster run.
%cd ml
!python src/train_behavioral.py --users 10000 --kp-epochs 20 --mouse-epochs 15
%cd ..

In [ ]:
# 4. Review measured metrics
import json
print(json.dumps(json.load(open('ml/reports/metrics_behavioral.json')), indent=2))

In [ ]:
# 5. Download the 3 trained weights (drop into ml/models/ in the repo)
from google.colab import files
for w in ['behaveformer_finetuned.pth','siamese_head.pth','cnn_lstm.pth']:
    files.download(f'ml/models/{w}')